In [ ]:
# LLM Quest — Interface de Reformulação Epistêmica
# Execute esta célula e a seguinte. A interface abre logo abaixo.

In [3]:
import os, sys, warnings
warnings.filterwarnings('ignore')

project_root = os.path.abspath('')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(override=True)

os.environ['INFERENCE_BACKEND'] = 'claude'

In [4]:
import gradio as gr

_initialized = False

def _init():
    global _initialized
    if _initialized:
        return
    from src.rl.inference import _get_index
    _get_index()
    _initialized = True


def reformular_ui(pergunta, idioma):
    if not pergunta.strip():
        return '<p style="color:orange">Digite uma pergunta.</p>'

    _init()

    from src.rl.inference import reformular, reformular_ptbr
    ptbr = (idioma == 'Português')

    try:
        r = reformular_ptbr(pergunta, n=8) if ptbr else reformular(pergunta, n=8)
    except Exception as e:
        return f'<p style="color:red">Erro: {e}</p>'

    if ptbr:
        entrada, saida = r.q_bad_pt, r.best_pt
        sub_in  = f'<div style="color:#aaa;font-size:.82em;margin-top:3px">🔤 {r.q_bad_en}</div>'
        sub_out = f'<div style="color:#aaa;font-size:.82em;margin-top:3px">🔤 {r.best_en}</div>'
        cands, ee_bad, ee_best, score, passed = r.candidates, r.ee_bad, r.ee_best, r.score_best, r.stage1_pass
    else:
        entrada, saida = r.q_bad, r.best
        sub_in = sub_out = ''
        cands, ee_bad, ee_best, score, passed = r.candidates, r.ee_bad, r.ee_best, r.score_best, r.stage1_pass

    delta     = ee_best - ee_bad
    ganho_pct = delta / max(ee_bad, 0.001) * 100
    filtro    = '✅ PASS' if passed else '⚠️ Fallback'
    fc        = '#27ae60' if passed else '#e67e22'

    def bar(v):
        pct = min(int(v * 100), 100)
        c   = '#2ecc71' if pct >= 70 else '#f39c12' if pct >= 40 else '#e74c3c'
        return (f'<div style="display:inline-flex;align-items:center;gap:6px">'
                f'<div style="background:#ddd;border-radius:3px;height:8px;width:120px">'
                f'<div style="background:{c};width:{pct}%;height:100%;border-radius:3px"></div></div>'
                f'<code style="font-size:.82em">{v:.3f}</code></div>')

    top  = sorted(cands, key=lambda c: c['score'], reverse=True)
    rows = ''.join(
        f'<tr style="{"font-weight:600;background:#f0fff4" if i==1 else ""};border-bottom:1px solid #eee">'
        f'<td style="padding:4px 8px;text-align:center">{"🟢" if c["ee"]>ee_bad+0.05 else "🔴"} {i}</td>'
        f'<td style="padding:4px 8px;font-family:monospace">{c["ee"]:.3f}</td>'
        f'<td style="padding:4px 8px">{c["text"]}</td></tr>'
        for i, c in enumerate(top, 1)
    )

    return f"""
<div style="font-family:system-ui,sans-serif;line-height:1.5;max-width:860px">
  <div style="background:#f5f5f5;border-left:4px solid #bbb;padding:10px 14px;border-radius:4px;margin-bottom:10px">
    <div style="font-size:.72em;color:#888;text-transform:uppercase">Pergunta original</div>
    <div style="margin-top:4px">{entrada}</div>{sub_in}
  </div>
  <div style="background:#eafaf1;border-left:4px solid #2ecc71;padding:10px 14px;border-radius:4px;margin-bottom:14px">
    <div style="font-size:.72em;color:#27ae60;text-transform:uppercase">Reformulação epistêmica</div>
    <div style="font-weight:600;font-size:1.04em;margin-top:4px">{saida}</div>{sub_out}
  </div>
  <div style="display:flex;gap:28px;flex-wrap:wrap;margin-bottom:14px;font-size:.9em">
    <div><div style="font-size:.72em;color:#888">EE ANTES</div>{bar(ee_bad)}</div>
    <div><div style="font-size:.72em;color:#888">EE DEPOIS</div>{bar(ee_best)}</div>
    <div><div style="font-size:.72em;color:#888">GANHO</div>
      <span style="font-weight:700;color:#27ae60">+{delta:.3f} ({ganho_pct:.0f}%)</span></div>
    <div><div style="font-size:.72em;color:#888">FILTRO</div>
      <span style="color:{fc};font-weight:600">{filtro}</span></div>
  </div>
  <details>
    <summary style="cursor:pointer;color:#2980b9;font-size:.85em">▶ Ver {len(cands)} candidatos</summary>
    <table style="border-collapse:collapse;width:100%;margin-top:6px;font-size:.82em">
      <thead><tr style="background:#f0f0f0">
        <th style="padding:4px 8px">#</th>
        <th style="padding:4px 8px">EE</th>
        <th style="padding:4px 8px;text-align:left">Reformulação</th>
      </tr></thead>
      <tbody>{rows}</tbody>
    </table>
  </details>
</div>
"""


with gr.Blocks(title='LLM Quest', theme=gr.themes.Soft()) as app:

    gr.Markdown("## 🔬 LLM Quest — Reformulação Epistêmica")

    with gr.Row():
        with gr.Column(scale=3):
            inp_q = gr.Textbox(
                label='Pergunta de pesquisa',
                placeholder='Ex: O que causa o envelhecimento biológico?',
                lines=3
            )
        with gr.Column(scale=1):
            inp_idioma = gr.Radio(
                choices=['Português', 'English'],
                value='Português',
                label='Idioma'
            )

    btn = gr.Button('🔄 Reformular', variant='primary', size='lg')
    out = gr.HTML()

    btn.click(fn=reformular_ui, inputs=[inp_q, inp_idioma], outputs=out)

    gr.Examples(
        examples=[
            ['O que causa o envelhecimento biológico?',           'Português'],
            ['O livre-arbítrio existe?',                          'Português'],
            ['Qual é a natureza da consciência?',                 'Português'],
            ['What is the nature of dark matter?',                'English'  ],
            ['Does consciousness arise from physical processes?', 'English'  ],
        ],
        inputs=[inp_q, inp_idioma],
        label='Exemplos'
    )

app.launch(inline=True, share=False, quiet=True)

Indice BM25 carregado do cache.
  [1/3] Traduzindo entrada (pt -> en)...
  -> What is the impact of ancestry?
  [2/3] Reformulando (pipeline EE)...
